# TPI IA 2026 — Comprensión visual de escenas

**Competencia:** `tpi-ia-2026-primavera-comprension-visual-de-escenas` (Kaggle, Cátedra Inteligencia Artificial, UTN)

**Integrantes:** _(completar)_ · **Nro. de grupo:** _(completar)_

Esta notebook implementa de punta a punta la pipeline *imagen → detección → segmentación → SceneJSON → preguntas (determinista + LLM)*, siguiendo las 9 etapas de "Desarrollo del trabajo" del enunciado. El reporte técnico completo (decisiones, experimentos, análisis de errores) está en `REPORTE.md` del repositorio del grupo; acá se resume lo esencial de cada decisión junto al código real que la implementa.

**Reglas respetadas:** las anotaciones del dataset de desarrollo se usan solo para EDA y evaluación (nunca para asistir la inferencia); el LLM nunca recibe la imagen ni los campos estructurados reservados al modo determinista; toda la pipeline corre sin archivos externos propios, solo el dataset oficial de la competencia y pesos de modelos públicos descargados por la propia notebook.

## Setup

Instala dependencias que no vienen preinstaladas en el entorno de Kaggle, detecta dónde quedó montado el dataset de la competencia, fija semillas y detecta GPU.

In [ ]:
!pip install -q -U transformers scipy jsonschema

In [ ]:
import json
import random
import time
from pathlib import Path

import numpy as np
import torch

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

In [ ]:
# El dataset queda montado automáticamente al crear la notebook desde la
# competencia (pestaña Data -> Add competition data), sin usar Add Input
# manual. Se busca con un fallback por si el nombre exacto de la carpeta
# cambia entre entornos.
CANDIDATE_ROOTS = [
    Path("/kaggle/input/tpi-ia-2026-primavera-comprension-visual-de-escenas"),
]
DATA_ROOT = next((p for p in CANDIDATE_ROOTS if (p / "scene_schema.json").exists()), None)

if DATA_ROOT is None and Path("/kaggle/input").exists():
    for candidate in Path("/kaggle/input").iterdir():
        if (candidate / "scene_schema.json").exists():
            DATA_ROOT = candidate
            break

if DATA_ROOT is None:
    raise FileNotFoundError(
        "No se encontro el dataset de la competencia en /kaggle/input/. "
        "Verificar que la notebook se haya creado desde la competencia "
        "(pestana Data -> Add competition data)."
    )

print(f"Dataset encontrado en: {DATA_ROOT}")

DEVELOPMENT_DIR = DATA_ROOT / "development"
TEST_DIR = DATA_ROOT / "tpi_ia_2026_competition_public"
QUESTIONS_PATH = DATA_ROOT / "tpi_2026_qa_public" / "questions_dev.jsonl"
SAMPLE_SUBMISSION_PATH = DATA_ROOT / "sample_submission.csv"
SCHEMA_PATH = DATA_ROOT / "scene_schema.json"

OUTPUT_DIR = Path("/kaggle/working")
TRACE_PATH = OUTPUT_DIR / "trace" / "pipeline.jsonl"

# Umbral de confianza baseline (ver justificacion en la seccion de Deteccion).
# Cambiar a 0.3 para reproducir la comparacion controlada en lugar del baseline.
CONFIDENCE_THRESHOLD = 0.1
NMS_IOU_THRESHOLD = 0.5

# Poner en False para saltear el experimento controlado (umbral 0.3) y
# ahorrar tiempo en una corrida de prueba rapida.
RUN_CONTROLLED_EXPERIMENT = True

## Etapa 1 — Análisis exploratorio del dataset

El EDA no es una colección mecánica de gráficos: cada número presentado motivó una decisión posterior concreta.

- **Distribución de categorías** (cola larga: 46 instancias en la más frecuente vs. 3 en las más raras) → no tiene sentido afinar el umbral de confianza por categoría, se usa uno global.
- **`area_ratio`** (mediana ≈0.003 — los objetos son muy chicos) → justifica un umbral de confianza permisivo en detección, y define los cortes de `size` (terciles empíricos, no números redondos inventados).
- **Oclusión** (13% de instancias con IoU>0.3 con su vecino) → anticipa la necesidad de NMS por categoría en detección y de limitar las relaciones espaciales a vecinos cercanos.
- **`neg_category_ids`/`not_exhaustive_category_ids`** → define directamente la política LVIS-aware de las métricas de detección/máscaras.

In [ ]:
def bbox_to_corners(bbox: list[float]) -> tuple[float, float, float, float]:
    x, y, w, h = bbox
    return x, y, x + w, y + h


def bbox_iou(bbox_a: list[float], bbox_b: list[float]) -> float:
    ax1, ay1, ax2, ay2 = bbox_to_corners(bbox_a)
    bx1, by1, bx2, by2 = bbox_to_corners(bbox_b)

    inter_w = max(0.0, min(ax2, bx2) - max(ax1, bx1))
    inter_h = max(0.0, min(ay2, by2) - max(ay1, by1))
    inter_area = inter_w * inter_h
    if inter_area == 0:
        return 0.0

    area_a = (ax2 - ax1) * (ay2 - ay1)
    area_b = (bx2 - bx1) * (by2 - by1)
    return inter_area / (area_a + area_b - inter_area)

In [ ]:
import json
from itertools import combinations
from pathlib import Path

import pandas as pd



def load_annotations(development_dir: Path) -> dict:
    with open(Path(development_dir) / "annotations.json", "r", encoding="utf-8") as f:
        return json.load(f)


def category_distribution(annotations: dict) -> pd.DataFrame:
    cat_names = {c["id"]: c["name"] for c in annotations["categories"]}
    ann_df = pd.DataFrame(annotations["annotations"])

    num_annotations = ann_df.groupby("category_id").size()
    num_images = ann_df.groupby("category_id")["image_id"].nunique()

    df = pd.DataFrame({
        "category_id": num_annotations.index,
        "name": [cat_names[c] for c in num_annotations.index],
        "num_annotations": num_annotations.values,
        "num_images": num_images.reindex(num_annotations.index).values,
    })
    return df.sort_values("num_annotations", ascending=False).reset_index(drop=True)


def instances_per_image(annotations: dict) -> pd.Series:
    ann_df = pd.DataFrame(annotations["annotations"])
    counts = ann_df.groupby("image_id").size()
    all_image_ids = [img["id"] for img in annotations["images"]]
    return counts.reindex(all_image_ids, fill_value=0).rename("num_instances")


def object_size_distribution(annotations: dict) -> pd.DataFrame:
    img_dims = {img["id"]: img["width"] * img["height"] for img in annotations["images"]}
    rows = []
    for ann in annotations["annotations"]:
        image_area = img_dims[ann["image_id"]]
        _, _, w, h = ann["bbox"]
        rows.append({
            "annotation_id": ann.get("id"),
            "image_id": ann["image_id"],
            "category_id": ann["category_id"],
            "area": ann.get("area", w * h),
            "area_ratio": ann.get("area", w * h) / image_area,
            "bbox_w": w,
            "bbox_h": h,
        })
    return pd.DataFrame(rows)


def occlusion_stats(annotations: dict) -> pd.DataFrame:
    ann_df = pd.DataFrame(annotations["annotations"])
    rows = []
    for image_id, group in ann_df.groupby("image_id"):
        boxes = group["bbox"].tolist()
        ann_ids = group["id"].tolist() if "id" in group else list(range(len(boxes)))
        max_iou = {aid: 0.0 for aid in ann_ids}
        for (i, bbox_a), (j, bbox_b) in combinations(enumerate(boxes), 2):
            iou = bbox_iou(bbox_a, bbox_b)
            if iou > max_iou[ann_ids[i]]:
                max_iou[ann_ids[i]] = iou
            if iou > max_iou[ann_ids[j]]:
                max_iou[ann_ids[j]] = iou
        for aid, iou in max_iou.items():
            rows.append({"image_id": image_id, "annotation_id": aid, "max_neighbor_iou": iou})
    return pd.DataFrame(rows)


def negative_and_non_exhaustive_categories(annotations: dict) -> dict:
    cat_names = {c["id"]: c["name"] for c in annotations["categories"]}

    neg_counts: dict[int, int] = {}
    not_exhaustive_counts: dict[int, int] = {}
    images_with_neg = 0
    images_with_not_exhaustive = 0

    for img in annotations["images"]:
        neg_ids = img.get("neg_category_ids", [])
        not_exh_ids = img.get("not_exhaustive_category_ids", [])
        if neg_ids:
            images_with_neg += 1
        if not_exh_ids:
            images_with_not_exhaustive += 1
        for cid in neg_ids:
            neg_counts[cid] = neg_counts.get(cid, 0) + 1
        for cid in not_exh_ids:
            not_exhaustive_counts[cid] = not_exhaustive_counts.get(cid, 0) + 1

    return {
        "images_with_neg_category_ids": images_with_neg,
        "images_with_not_exhaustive_category_ids": images_with_not_exhaustive,
        "neg_category_counts": {
            cat_names[cid]: n for cid, n in sorted(neg_counts.items(), key=lambda kv: -kv[1])
        },
        "not_exhaustive_category_counts": {
            cat_names[cid]: n for cid, n in sorted(not_exhaustive_counts.items(), key=lambda kv: -kv[1])
        },
    }

In [ ]:
annotations = load_annotations(DEVELOPMENT_DIR)
categories = annotations["categories"]

categories_df = category_distribution(annotations)
print(categories_df.head(10).to_string(index=False))

instances = instances_per_image(annotations)
print(f"instancias por imagen: media={instances.mean():.1f} mediana={instances.median():.0f} max={instances.max()}")

sizes = object_size_distribution(annotations)
print(f"area_ratio terciles: {sizes['area_ratio'].quantile(1/3):.5f} / {sizes['area_ratio'].quantile(2/3):.5f}")

occlusion = occlusion_stats(annotations)
print(f"instancias con IoU>0.3 con su vecino: {(occlusion['max_neighbor_iou']>0.3).mean()*100:.1f}%")

neg_summary = negative_and_non_exhaustive_categories(annotations)
print(f"imagenes con neg_category_ids: {neg_summary['images_with_neg_category_ids']}")
print(f"imagenes con not_exhaustive_category_ids: {neg_summary['images_with_not_exhaustive_category_ids']}")

## Etapa 2 — Detección open-vocabulary

**Detector: OWLv2** (`google/owlv2-base-patch16-ensemble`, vía `transformers`). Elegido sobre Grounding DINO por ser pip-instalable sin compilar operadores CUDA custom, más confiable en un entorno compartido como Kaggle. Soporta vocabulario de texto libre.

- **Vocabulario:** las 120 categorías completas del dataset, en un solo batch de texto por imagen (no sabemos de antemano qué categorías aparecen en el test).
- **Umbral de confianza: 0.1** (permisivo), justificado por el EDA: objetos chicos dan confianza baja incluso en aciertos reales.
- **Resolución de entrada: 960×960 fija**, la nativa del checkpoint (resize no-uniforme, sin letterbox, verificado en el código fuente de `transformers`). No se la modifica porque el backbone fue entrenado específicamente a ese tamaño — cambiarla arriesga más de lo que soluciona sin re-calibración.
- **Supresión de duplicados: NMS por categoría** (no global), para no borrar objetos distintos que se superponen legítimamente.

In [ ]:
import torch
from PIL import Image
from transformers import Owlv2ForObjectDetection, Owlv2Processor


_DET_MODEL_NAME = "google/owlv2-base-patch16-ensemble"
_det_processor = None
_det_model = None


def _get_det_model():
    global _det_processor, _det_model
    if _det_model is None:
        _det_processor = Owlv2Processor.from_pretrained(_DET_MODEL_NAME)
        _det_model = Owlv2ForObjectDetection.from_pretrained(_DET_MODEL_NAME).to(DEVICE)
        _det_model.eval()
    return _det_processor, _det_model


def detect_objects(
    image: Image.Image,
    vocabulary: dict[str, int],
    confidence_threshold: float,
) -> list[dict]:
    """vocabulary maps category name -> category_id (from annotations.json
    categories). Returns a list of {category_id, confidence, bbox} (bbox in
    xywh) before NMS/dedup."""
    processor, model = _get_det_model()
    names = list(vocabulary.keys())

    inputs = processor(text=[names], images=image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)

    target_sizes = torch.tensor([image.size[::-1]]).to(DEVICE)
    results = processor.post_process_grounded_object_detection(
        outputs=outputs,
        threshold=confidence_threshold,
        target_sizes=target_sizes,
        text_labels=[names],
    )[0]

    detections = []
    for score, box, name in zip(results["scores"], results["boxes"], results["text_labels"]):
        x1, y1, x2, y2 = box.cpu().tolist()
        detections.append({
            "category_id": vocabulary[name],
            "confidence": float(score),
            "bbox": [x1, y1, x2 - x1, y2 - y1],
        })
    return detections


def suppress_duplicates(detections: list[dict], iou_threshold: float) -> list[dict]:
    """Per-category NMS: keeps the highest-confidence box and drops any
    same-category box overlapping it above iou_threshold."""
    by_category: dict[int, list[dict]] = {}
    for det in detections:
        by_category.setdefault(det["category_id"], []).append(det)

    kept = []
    for dets in by_category.values():
        dets = sorted(dets, key=lambda d: d["confidence"], reverse=True)
        chosen: list[dict] = []
        for det in dets:
            if all(bbox_iou(det["bbox"], c["bbox"]) <= iou_threshold for c in chosen):
                chosen.append(det)
        kept.extend(chosen)
    return kept

In [ ]:
from PIL import Image

VOCABULARY = {c["name"]: c["id"] for c in categories}


def detect_all(image_ids: list[str], image_dir: Path, file_name_fn, tracer=None) -> list[dict]:
    """Runs detect_objects + suppress_duplicates over every image_id,
    tagging predictions with "image_id". file_name_fn(image_id) -> filename."""
    predictions = []
    t0 = time.time()
    for i, image_id in enumerate(image_ids):
        image = Image.open(image_dir / file_name_fn(image_id)).convert("RGB")
        raw = detect_objects(image, VOCABULARY, confidence_threshold=CONFIDENCE_THRESHOLD)
        kept = suppress_duplicates(raw, iou_threshold=NMS_IOU_THRESHOLD)
        for det in kept:
            det["image_id"] = image_id
        predictions.extend(kept)
        if (i + 1) % 20 == 0 or i == len(image_ids) - 1:
            print(f"  [{i+1}/{len(image_ids)}] {time.time()-t0:.0f}s, {len(predictions)} predicciones acumuladas")
    return predictions

Detección sobre las 120 imágenes de desarrollo (para evaluación con ground truth):

In [ ]:
dev_image_ids = [img["id"] for img in annotations["images"]]
dev_file_names = {img["id"]: img["file_name"] for img in annotations["images"]}

dev_predictions = detect_all(
    dev_image_ids, DEVELOPMENT_DIR / "images", lambda iid: dev_file_names[iid]
)
print(f"Total predicciones (desarrollo): {len(dev_predictions)}")

## Etapa 3 — Segmentación de instancias con SAM2 + propiedades derivadas

**Segmentador: SAM2.1 tiny** (`facebook/sam2.1-hiera-tiny`, vía `transformers` — misma lógica que con el detector, evita clonar/compilar el repo oficial de Meta). Las cajas detectadas se usan como *prompts*, todas las de una imagen en un único forward batcheado.

**Propiedades derivadas de las máscaras** (`centroid`, `area_ratio`, `position`, `size`, `dominant_color`) y **relaciones espaciales** (`compute_relations`): limitadas a los 5 vecinos más cercanos por objeto, no todos los pares — la primera versión sin ese límite generaba más de 12000 relaciones en la imagen más poblada, mayormente ruido entre objetos lejanos sin relación real.

In [ ]:
import numpy as np
import torch
from PIL import Image
from transformers import Sam2Model, Sam2Processor

_SAM_MODEL_NAME = "facebook/sam2.1-hiera-tiny"
_sam_processor = None
_sam_model = None


def _get_sam_model():
    global _sam_processor, _sam_model
    if _sam_model is None:
        _sam_processor = Sam2Processor.from_pretrained(_SAM_MODEL_NAME)
        _sam_model = Sam2Model.from_pretrained(_SAM_MODEL_NAME).to(DEVICE)
        _sam_model.eval()
    return _sam_processor, _sam_model


def segment_with_sam2(image: Image.Image, boxes: list[list[float]]) -> list[np.ndarray]:
    """boxes: list of [x, y, w, h] (same convention as detection.detect_objects).
    Returns one boolean HxW mask per box, same order as boxes, same size as image."""
    if not boxes:
        return []

    processor, model = _get_sam_model()
    boxes_xyxy = [[x, y, x + w, y + h] for x, y, w, h in boxes]

    inputs = processor(images=image, input_boxes=[boxes_xyxy], return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs, multimask_output=False)

    masks = processor.post_process_masks(outputs.pred_masks, inputs["original_sizes"])[0]
    # masks: (num_boxes, 1, H, W) bool tensor
    return [masks[i, 0].cpu().numpy() for i in range(masks.shape[0])]


def mask_to_rle(mask: np.ndarray) -> dict:
    """Uncompressed RLE, column-major (Fortran) order, per scene_schema.json:
    {size: [h, w], counts: [int, ...]} with counts alternating background/
    foreground run-lengths, starting with background (can be 0)."""
    h, w = mask.shape
    flat = mask.flatten(order="F").astype(np.uint8)

    counts: list[int] = []
    prev = 0
    run_length = 0
    for value in flat:
        value = int(value)
        if value == prev:
            run_length += 1
        else:
            counts.append(run_length)
            prev = value
            run_length = 1
    counts.append(run_length)

    return {"size": [h, w], "counts": counts}


def rle_to_mask(rle: dict) -> np.ndarray:
    h, w = rle["size"]
    flat = np.zeros(h * w, dtype=bool)
    pos = 0
    value = False
    for run_length in rle["counts"]:
        if value:
            flat[pos:pos + run_length] = True
        pos += run_length
        value = not value
    return flat.reshape((h, w), order="F")

In [ ]:
import numpy as np
from PIL import Image


POSITIONS = [
    "top_left", "top_center", "top_right",
    "middle_left", "center", "middle_right",
    "bottom_left", "bottom_center", "bottom_right",
]
SIZES = ["small", "medium", "large"]
PREDICATES = ["left_of", "right_of", "above", "below", "overlaps"]

# Empirical tertiles of area_ratio over the 1790 real annotations of the
# development set (reports/eda/object_sizes.csv), not arbitrary round
# numbers: ensures small/medium/large are roughly balanced on this dataset.
SMALL_MEDIUM_THRESHOLD = 0.00126
MEDIUM_LARGE_THRESHOLD = 0.00765

# Minimum bbox IoU to call two objects "overlaps" rather than ignoring
# the pair; avoids tagging near-zero grazing overlaps as a relation.
OVERLAP_IOU_THRESHOLD = 0.01


def compute_centroid(mask: np.ndarray) -> list[float]:
    ys, xs = np.nonzero(mask)
    if len(xs) == 0:
        return [0.0, 0.0]
    return [float(xs.mean()), float(ys.mean())]


def compute_area_ratio(mask: np.ndarray, image_width: int, image_height: int) -> float:
    return float(mask.sum()) / (image_width * image_height)


def compute_position(centroid: list[float], image_width: int, image_height: int) -> str:
    cx, cy = centroid
    col = min(2, int(cx / image_width * 3))
    row = min(2, int(cy / image_height * 3))
    return POSITIONS[row * 3 + col]


def compute_size(area_ratio: float) -> str:
    if area_ratio < SMALL_MEDIUM_THRESHOLD:
        return "small"
    if area_ratio < MEDIUM_LARGE_THRESHOLD:
        return "medium"
    return "large"


def compute_dominant_color(image: Image.Image, mask: np.ndarray, bin_size: int = 24) -> tuple[int, int, int]:
    """Quantizes masked pixels into bin_size-wide RGB bins and returns the
    center of the most frequent bin (the actual dominant color, not just
    a central-tendency average that could land outside any real color)."""
    arr = np.array(image.convert("RGB"))
    pixels = arr[mask]
    if len(pixels) == 0:
        return (0, 0, 0)

    quantized = (pixels // bin_size) * bin_size + bin_size // 2
    colors, counts = np.unique(quantized, axis=0, return_counts=True)
    dominant = colors[np.argmax(counts)]
    return tuple(int(c) for c in dominant)


NEAREST_NEIGHBORS = 5


def _bbox_center(bbox: list[float]) -> tuple[float, float]:
    x1, y1, x2, y2 = bbox_to_corners(bbox)
    return (x1 + x2) / 2, (y1 + y2) / 2


def compute_relations(objects: list[dict], k: int = NEAREST_NEIGHBORS) -> list[dict]:
    """objects: list of {id, bbox} (bbox in xywh) for a single image.
    Returns [{subject, predicate, object}, ...] using object ids.

    Relations are only computed between each object and its k nearest
    neighbors (by centroid distance), not all O(n^2) pairs: on a crowded
    image (e.g. 257 objects) "left_of"/"above" between every axis-aligned
    pair produced >12000 near-meaningless relations (unrelated objects on
    opposite sides of the frame that happen to share a row/column). Capping
    at k nearest neighbors keeps relations tied to objects that are
    actually close to each other, which is what the "útil" requirement in
    the assignment is after.

    For each kept pair:
    - "overlaps" when bbox IoU > OVERLAP_IOU_THRESHOLD.
    - otherwise "left_of" when the pair's vertical (y) ranges intersect
      (roughly the same height), oriented so subject has the smaller
      center x.
    - otherwise "above" when the pair's horizontal (x) ranges intersect
      (roughly the same column), oriented so subject has the smaller
      center y (image y grows downward).
    Only one canonical direction is stored per axis (left_of, above); the
    inverse (right_of, below) is implied and left for the QA engine to
    check both directions when asked."""
    n = len(objects)
    if n < 2:
        return []

    centers = [_bbox_center(o["bbox"]) for o in objects]
    pairs: set[tuple[int, int]] = set()
    for i in range(n):
        distances = sorted(
            (
                (centers[i][0] - centers[j][0]) ** 2 + (centers[i][1] - centers[j][1]) ** 2,
                j,
            )
            for j in range(n) if j != i
        )
        for _, j in distances[:k]:
            pairs.add((i, j) if i < j else (j, i))

    relations = []
    for i, j in sorted(pairs):
        a, b = objects[i], objects[j]
        iou = bbox_iou(a["bbox"], b["bbox"])
        if iou > OVERLAP_IOU_THRESHOLD:
            relations.append({"subject": a["id"], "predicate": "overlaps", "object": b["id"]})
            continue

        ax1, ay1, ax2, ay2 = bbox_to_corners(a["bbox"])
        bx1, by1, bx2, by2 = bbox_to_corners(b["bbox"])
        y_overlap = ay1 < by2 and by1 < ay2
        x_overlap = ax1 < bx2 and bx1 < ax2

        if y_overlap:
            acx, bcx = (ax1 + ax2) / 2, (bx1 + bx2) / 2
            left, right = (a, b) if acx < bcx else (b, a)
            relations.append({"subject": left["id"], "predicate": "left_of", "object": right["id"]})
        if x_overlap:
            acy, bcy = (ay1 + ay2) / 2, (by1 + by2) / 2
            top, bottom = (a, b) if acy < bcy else (b, a)
            relations.append({"subject": top["id"], "predicate": "above", "object": bottom["id"]})
    return relations

In [ ]:
def segment_and_enrich(image_id: str, image_path: Path, boxes_dets: list[dict]) -> dict:
    """boxes_dets: detections for one image, as produced by detect_all
    (before grouping). Returns the full SceneJSON for that image (image +
    objects with all derived properties + relations), still missing
    questions/questions_llm."""
    image = Image.open(image_path).convert("RGB")
    w, h = image.size

    objs_for_scene = []
    if boxes_dets:
        masks = segment_with_sam2(image, [d["bbox"] for d in boxes_dets])
        for det, mask in zip(boxes_dets, masks):
            objs_for_scene.append({
                "category_id": det["category_id"],
                "confidence": det["confidence"],
                "bbox": det["bbox"],
                "mask": mask_to_rle(mask),
            })

    scene = build_scene_json(image_id, w, h, objs_for_scene)

    enriched_objects = []
    for obj in scene["objects"]:
        mask = rle_to_mask(obj["mask"])
        centroid = compute_centroid(mask)
        area_ratio = compute_area_ratio(mask, w, h)
        color = compute_dominant_color(image, mask)
        enriched_objects.append({
            **obj, "centroid": centroid, "area_ratio": area_ratio,
            "position": compute_position(centroid, w, h),
            "size": compute_size(area_ratio),
            "dominant_color": list(color),
        })
    scene["objects"] = enriched_objects
    scene["relations"] = compute_relations([{"id": o["id"], "bbox": o["bbox"]} for o in enriched_objects])
    return scene

## Etapa 4 — Representación estructurada de la escena (`SceneJSON`)

`build_scene_json` ensambla el esqueleto (`image`, `objects`, `relations`) con `id` único por objeto. Nota de diseño: el ensamblado es en dos pasos — esta función arma la estructura, `properties.py` (arriba) calcula las propiedades geométricas — por eso `segment_and_enrich` combina ambos pasos para dejar cada escena completa de una.

Se construyen las escenas de **desarrollo** (para evaluación) y de **test** (para el envío final).

In [ ]:
def build_scene_json(
    image_id: str,
    width: int,
    height: int,
    objects: list[dict],
) -> dict:
    """objects: list of {category_id, confidence, bbox, mask} (e.g. from
    segmentation.segment_with_sam2 + mask_to_rle, one per detected+segmented
    instance). Assigns each object a unique string id.

    This only assembles the structural skeleton (image, objects, relations)
    required by scene_schema.json: centroid/area_ratio/position/size are
    NOT computed here (src/properties.py fills those into each object
    afterward) and relations starts empty (filled by
    properties.compute_relations once objects have centroids/bboxes)."""
    scene_objects = []
    for i, obj in enumerate(objects, start=1):
        scene_objects.append({
            "id": f"{image_id}_obj{i}",
            "category_id": obj["category_id"],
            "confidence": obj["confidence"],
            "bbox": obj["bbox"],
            "mask": obj["mask"],
        })

    return {
        "image": {"id": image_id, "width": width, "height": height},
        "objects": scene_objects,
        "relations": [],
    }

In [ ]:
print("Construyendo escenas de desarrollo (detección + segmentación + propiedades)...")
t0 = time.time()

dev_preds_by_image: dict[str, list[dict]] = {}
for pred in dev_predictions:
    dev_preds_by_image.setdefault(pred["image_id"], []).append(pred)

dev_scenes: dict[str, dict] = {}
for i, img_info in enumerate(annotations["images"]):
    image_id = Path(img_info["file_name"]).stem
    scene = segment_and_enrich(
        image_id,
        DEVELOPMENT_DIR / "images" / img_info["file_name"],
        dev_preds_by_image.get(img_info["id"], []),
    )
    dev_scenes[image_id] = scene
    if (i + 1) % 20 == 0 or i == len(annotations["images"]) - 1:
        print(f"  [{i+1}/{len(annotations['images'])}] {time.time()-t0:.0f}s")

print(f"Escenas de desarrollo construidas: {len(dev_scenes)}")

In [ ]:
import jsonschema

schema = json.load(open(SCHEMA_PATH))
validator = jsonschema.Draft202012Validator(schema)
invalid = [iid for iid, s in dev_scenes.items() if list(validator.iter_errors(s))]
print(f"Escenas validas contra el schema: {len(dev_scenes)-len(invalid)}/{len(dev_scenes)}")
if invalid:
    print("Invalidas:", invalid[:5])

## Etapa 5 — Trazabilidad de la pipeline

`Tracer` escribe una línea JSON por paso (formato *JSON Lines*, auditable) en `/kaggle/working/trace/pipeline.jsonl`, con `timestamp`, `run_id`, `image_id`, `module`, `status` y `duration_seconds`. `Tracer.step(...)` es un *context manager* que, si el bloque falla, registra `status="error"` con el mensaje de la excepción **y la vuelve a lanzar** — un paso que falla sigue fallando de forma visible, pero queda localizado en la traza.

A partir de acá se usa `tracer.step(...)` alrededor de cada llamada a las funciones ya definidas arriba, para dejar trazada la ejecución real de las etapas 6-9.

In [ ]:
import json
import time
import uuid
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path


class Tracer:
    """Appends one JSON line per step to output_path (JSON Lines: easy to
    audit, grep, and reload without parsing the whole file as one JSON
    document). Every record carries a run_id so traces from different runs
    can share a file without being confused for each other."""

    def __init__(self, output_path: Path):
        self.output_path = Path(output_path)
        self.output_path.parent.mkdir(parents=True, exist_ok=True)
        self.run_id = uuid.uuid4().hex[:8]

    def log_step(self, image_id: str, module: str, params: dict, status: str, **extra) -> None:
        record = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "run_id": self.run_id,
            "image_id": image_id,
            "module": module,
            "status": status,
            "params": params,
            **extra,
        }
        with open(self.output_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, default=str) + "\n")

    @contextmanager
    def step(self, image_id: str, module: str, params: dict | None = None):
        """Times the block and logs it automatically: status="ok" with
        duration_seconds on success, or status="error" with the exception
        message on failure (then re-raises, so the step still fails loudly
        -- it just also leaves a record of where it failed)."""
        start = time.time()
        try:
            yield
        except Exception as exc:
            self.log_step(image_id, module, params or {}, status="error",
                          duration_seconds=time.time() - start, error=str(exc))
            raise
        else:
            self.log_step(image_id, module, params or {}, status="ok",
                          duration_seconds=time.time() - start)


def load_trace(path: Path) -> list[dict]:
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

In [ ]:
tracer = Tracer(TRACE_PATH)
print(f"Tracer inicializado, run_id={tracer.run_id}, escribiendo en {TRACE_PATH}")

## Etapa 6 — Preguntas y respuestas, modo determinista

`answer_question` recibe una fila de `questions_dev.jsonl` (`type`, `class_en`, sin pasar por ningún LLM) y construye la respuesta consultando directamente el `SceneJSON`.

- **Predicado de `relation`:** no viene como campo estructurado separado, solo embebido en el texto — se resuelve con matching de frases fijas en español (`_PREDICATE_PATTERNS`), verificado contra las 10 preguntas `relation` reales del benchmark.
- **Ambigüedad:** `position`/`size`/`relation` presuponen una única instancia; si hay 0 o más de 1 match, la respuesta es `unsupported` en vez de adivinar.
- **`confidence`:** mínimo de las detecciones involucradas en la respuesta.
- **`description`:** plantilla determinista (no LLM) con las 5 categorías más frecuentes.

In [ ]:
import re
from collections import Counter


QUESTION_TYPES = ["description", "count", "existence", "position", "size", "relation"]

# Deterministic mode receives the question text too, but must not use an LLM
# to interpret it. For "relation" questions the predicate (above/left_of/...)
# is not a separate structured field from the cátedra -- only class_en/
# class_es (the objects) and type are. These are fixed-phrase keyword
# patterns over the known Spanish wording, not free-form NLU: reading a
# known pattern by code, not "interpreting" the question.
_PREDICATE_PATTERNS = [
    (re.compile(r"por encima de|arriba de", re.IGNORECASE), "above"),
    (re.compile(r"por debajo de|abajo de", re.IGNORECASE), "below"),
    (re.compile(r"a la izquierda de", re.IGNORECASE), "left_of"),
    (re.compile(r"a la derecha de", re.IGNORECASE), "right_of"),
    (re.compile(r"se superpone|superposici[oó]n|solapa", re.IGNORECASE), "overlaps"),
]


def _extract_predicate(question_text: str) -> str | None:
    for pattern, predicate in _PREDICATE_PATTERNS:
        if pattern.search(question_text):
            return predicate
    return None


def _find_by_category(scene_json: dict, category_id: int) -> list[dict]:
    return [o for o in scene_json["objects"] if o["category_id"] == category_id]


def _base_answer(question_record: dict) -> dict:
    return {
        "question_id": question_record["question_id"],
        "type": question_record["type"],
        "question": question_record["question"],
    }


def unsupported_answer(question_record: dict) -> dict:
    return {
        **_base_answer(question_record),
        "answer": "unsupported",
        "normalized_answer": "unsupported",
        "evidence": [],
        "confidence": 0.0,
    }


def _resolve_single_category(question_record: dict, name_to_id: dict[str, int], index: int = 0) -> int | None:
    names = question_record.get("class_en", [])
    if len(names) <= index or names[index] not in name_to_id:
        return None
    return name_to_id[names[index]]


def _answer_count(scene_json: dict, q: dict, name_to_id: dict[str, int]) -> dict:
    category_id = _resolve_single_category(q, name_to_id)
    if category_id is None:
        return unsupported_answer(q)
    matches = _find_by_category(scene_json, category_id)
    n = len(matches)
    confidence = min((o["confidence"] for o in matches), default=1.0)
    return {
        **_base_answer(q),
        "answer": str(n),
        "normalized_answer": n,
        "evidence": [o["id"] for o in matches],
        "confidence": confidence,
    }


def _answer_existence(scene_json: dict, q: dict, name_to_id: dict[str, int]) -> dict:
    category_id = _resolve_single_category(q, name_to_id)
    if category_id is None:
        return unsupported_answer(q)
    matches = _find_by_category(scene_json, category_id)
    exists = len(matches) > 0
    confidence = min((o["confidence"] for o in matches), default=1.0)
    normalized = "yes" if exists else "no"
    return {
        **_base_answer(q),
        "answer": normalized,
        "normalized_answer": normalized,
        "evidence": [o["id"] for o in matches],
        "confidence": confidence,
    }


def _answer_single_instance_field(scene_json: dict, q: dict, name_to_id: dict[str, int], field: str) -> dict:
    """Shared logic for "position"/"size": the question presupposes exactly
    one instance of the category ("la instancia de X"). Zero matches means
    the question doesn't apply; more than one is ambiguous (we can't know
    which instance is meant) -- both cases return unsupported rather than
    guessing."""
    category_id = _resolve_single_category(q, name_to_id)
    if category_id is None:
        return unsupported_answer(q)
    matches = _find_by_category(scene_json, category_id)
    if len(matches) != 1:
        return unsupported_answer(q)
    obj = matches[0]
    return {
        **_base_answer(q),
        "answer": obj[field],
        "normalized_answer": obj[field],
        "evidence": [obj["id"]],
        "confidence": obj["confidence"],
    }


def _evaluate_predicate(subject: dict, obj: dict, predicate: str) -> bool:
    if predicate == "overlaps":
        return bbox_iou(subject["bbox"], obj["bbox"]) > OVERLAP_IOU_THRESHOLD

    sx1, sy1, sx2, sy2 = bbox_to_corners(subject["bbox"])
    ox1, oy1, ox2, oy2 = bbox_to_corners(obj["bbox"])
    scx, scy = (sx1 + sx2) / 2, (sy1 + sy2) / 2
    ocx, ocy = (ox1 + ox2) / 2, (oy1 + oy2) / 2

    if predicate == "left_of":
        return scx < ocx
    if predicate == "right_of":
        return scx > ocx
    if predicate == "above":
        return scy < ocy
    if predicate == "below":
        return scy > ocy
    return False


def _answer_relation(scene_json: dict, q: dict, name_to_id: dict[str, int]) -> dict:
    subject_category = _resolve_single_category(q, name_to_id, index=0)
    object_category = _resolve_single_category(q, name_to_id, index=1)
    if subject_category is None or object_category is None:
        return unsupported_answer(q)

    subject_matches = _find_by_category(scene_json, subject_category)
    object_matches = _find_by_category(scene_json, object_category)
    if len(subject_matches) != 1 or len(object_matches) != 1:
        return unsupported_answer(q)

    # Deterministic-mode question records never carry "predicate" (the
    # cátedra's structured fields don't include it, see module docstring
    # above); LLM mode resolves it itself from the free-form question text
    # and passes it explicitly here, since the whole point of that mode is
    # NOT relying on fixed-phrase regex matching.
    predicate = q.get("predicate")
    if predicate not in PREDICATES:
        predicate = _extract_predicate(q["question"])
    if predicate is None:
        return unsupported_answer(q)

    subject, obj = subject_matches[0], object_matches[0]
    holds = _evaluate_predicate(subject, obj, predicate)
    normalized = "yes" if holds else "no"
    return {
        **_base_answer(q),
        "answer": normalized,
        "normalized_answer": normalized,
        "evidence": [subject["id"], obj["id"]],
        "confidence": min(subject["confidence"], obj["confidence"]),
    }


def _answer_description(scene_json: dict, q: dict, categories: list[dict]) -> dict:
    text = describe_scene(scene_json, categories)
    objects = scene_json["objects"]
    return {
        **_base_answer(q),
        "answer": text,
        "normalized_answer": text,
        "evidence": [o["id"] for o in objects],
        "confidence": min((o["confidence"] for o in objects), default=1.0),
    }


def answer_question(scene_json: dict, question_record: dict, categories: list[dict]) -> dict:
    """question_record: one row as provided by the cátedra (question_id,
    type, question, class_en, ...), e.g. from questions_dev.jsonl.
    categories: [{"id": int, "name": str}, ...] (same list as
    annotations.json["categories"]), used to resolve class_en to
    category_id. Returns a questionAnswer dict (question_id, type,
    question, answer, normalized_answer, evidence, confidence), with
    normalized_answer="unsupported" when the query can't be resolved
    from scene_json alone."""
    name_to_id = {c["name"]: c["id"] for c in categories}
    qtype = question_record["type"]

    if qtype == "description":
        return _answer_description(scene_json, question_record, categories)
    if qtype == "count":
        return _answer_count(scene_json, question_record, name_to_id)
    if qtype == "existence":
        return _answer_existence(scene_json, question_record, name_to_id)
    if qtype == "position":
        return _answer_single_instance_field(scene_json, question_record, name_to_id, "position")
    if qtype == "size":
        return _answer_single_instance_field(scene_json, question_record, name_to_id, "size")
    if qtype == "relation":
        return _answer_relation(scene_json, question_record, name_to_id)
    return unsupported_answer(question_record)


def describe_scene(scene_json: dict, categories: list[dict]) -> str:
    """Deterministic, template-based description (no LLM): the top-5 most
    frequent categories in the scene with their counts, in Spanish."""
    id_to_name = {c["id"]: c["name"] for c in categories}
    counts = Counter(o["category_id"] for o in scene_json["objects"])
    if not counts:
        return "No se detectaron objetos en la escena."

    top = counts.most_common(5)
    parts = [f"{n} {id_to_name.get(cid, f'categoria_{cid}')}" for cid, n in top]
    if len(parts) == 1:
        return f"La escena contiene {parts[0]}."
    return f"La escena contiene {', '.join(parts[:-1])} y {parts[-1]}."

In [ ]:
dev_questions = [json.loads(line) for line in open(QUESTIONS_PATH)]
print(f"Preguntas del benchmark: {len(dev_questions)}")

dev_answers_by_image: dict[str, list[dict]] = {}
for q in dev_questions:
    image_id = q["image_file"].rsplit(".", 1)[0]
    scene = dev_scenes[image_id]
    with tracer.step(image_id, "qa_engine", {"question_id": q["question_id"], "type": q["type"]}):
        answer = answer_question(scene, q, categories)
    dev_answers_by_image.setdefault(image_id, []).append(answer)

total_answers = sum(len(v) for v in dev_answers_by_image.values())
unsupported = sum(1 for v in dev_answers_by_image.values() for a in v if a["normalized_answer"] == "unsupported")
print(f"Respuestas generadas: {total_answers} ({unsupported} unsupported)")

## Etapa 7 — Preguntas y respuestas con LLM liviano (promoción)

**Modelo: Qwen2.5-1.5B-Instruct.** Se probó primero Qwen2.5-0.5B-Instruct (solo 22% de `class_en` exacto contra el vocabulario de 120 categorías — deja palabras en español sin traducir) antes de pasar a 1.5B (67.5%).

- **`interpret_query`** recibe *únicamente* el texto de la pregunta (nunca `type`/`class_en` de la cátedra) y produce un JSON intermedio `{type, class_en, predicate?}`, validado contra el vocabulario conocido antes de ejecutar — si no valida, `unsupported`, nunca se inventa una categoría.
- **Reutiliza el mismo motor determinista** (`answer_question`) para ejecutar la consulta validada — el LLM solo interpreta, nunca calcula la respuesta.
- **`generate_description_llm`** recibe solo pares categoría+conteo agregados, nunca la imagen. **Limitación medida y documentada:** no respeta de forma confiable la instrucción de no inventar información (ver `REPORTE.md` 4.8) — se reporta tal cual sale, sin maquillar.

In [ ]:
import json
import re
from collections import Counter

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


_LLM_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
_llm_tokenizer = None
_llm_model = None

_INTERPRET_SYSTEM_PROMPT = """Convertís una pregunta en español sobre una escena a un JSON de consulta estructurada. Respondé UNICAMENTE con el JSON en una sola linea, sin texto adicional, sin markdown.

Campos:
- "type": uno de ["description", "count", "existence", "position", "size", "relation"]
- "class_en": lista de nombres de categoria EN INGLES, copiados EXACTAMENTE de esta lista (nunca traduzcas vos, buscá el nombre que mas se parezca semánticamente): {categories}
- "predicate": solo si type es "relation", uno de ["left_of", "right_of", "above", "below", "overlaps"]

Ejemplos:
P: "Describí brevemente la escena."
{{"type": "description", "class_en": []}}

P: "¿Cuántas instancias de la categoría pomo de puerta aparecen en la imagen?"
{{"type": "count", "class_en": ["doorknob"]}}

P: "¿Hay al menos una instancia de tomacorriente en la imagen?"
{{"type": "existence", "class_en": ["wall socket"]}}

P: "¿En qué zona de la imagen está la instancia de teléfono celular?"
{{"type": "position", "class_en": ["cell phone"]}}

P: "¿Qué tamaño relativo tiene la instancia de collar?"
{{"type": "size", "class_en": ["necklace"]}}

P: "¿La instancia de reloj está por encima de la instancia de anillo?"
{{"type": "relation", "class_en": ["clock", "ring"], "predicate": "above"}}
"""

_DESCRIPTION_SYSTEM_PROMPT = (
    "Escribís una descripción breve y factual de una escena en español, usando "
    "EXCLUSIVAMENTE la lista de objetos detectados que te paso (categoría y "
    "cantidad). No inventes objetos, cantidades, colores ni ningún detalle que "
    "no esté en la lista. Respondé solo con la descripción, 1 o 2 oraciones, "
    "sin texto adicional."
)


def _get_llm_model():
    global _llm_tokenizer, _llm_model
    if _llm_model is None:
        _llm_tokenizer = AutoTokenizer.from_pretrained(_LLM_MODEL_NAME)
        _llm_model = AutoModelForCausalLM.from_pretrained(_LLM_MODEL_NAME).to(DEVICE)
        _llm_model.eval()
    return _llm_tokenizer, _llm_model


def _generate(system_prompt: str, user_prompt: str, max_new_tokens: int) -> str:
    tokenizer, model = _get_llm_model()
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def _extract_json(text: str) -> dict:
    """Finds the first {...} block anywhere in the output (the model
    sometimes wraps it in markdown fences or prose) rather than assuming
    the whole output is a bare single-line JSON object."""
    match = re.search(r"\{.*?\}", text, re.DOTALL)
    if not match:
        return {"type": None}
    try:
        parsed = json.loads(match.group(0))
    except json.JSONDecodeError:
        return {"type": None}
    return parsed if isinstance(parsed, dict) else {"type": None}


def interpret_query(question: str, categories: list[dict]) -> dict:
    """Receives only the question text (no type/class_en/class_es from the
    cátedra's structured fields -- those are reserved for deterministic
    mode). Returns the intermediate JSON {type, class_en, [predicate]},
    produced by the LLM, not validated yet (answer_question_llm validates
    it against the known category vocabulary before executing it)."""
    category_names = ", ".join(sorted(c["name"] for c in categories))
    system_prompt = _INTERPRET_SYSTEM_PROMPT.format(categories=category_names)
    raw = _generate(system_prompt, f'P: "{question}"', max_new_tokens=80)
    return _extract_json(raw)


def generate_description_llm(scene_json: dict, categories: list[dict]) -> str:
    """Generates the factual description from scene_json only, never from
    the image. The prompt only gets category+count pairs (not the raw
    object list, which could be hundreds of rows on a crowded image) with
    an explicit instruction not to invent anything beyond that list."""
    id_to_name = {c["id"]: c["name"] for c in categories}
    counts = Counter(o["category_id"] for o in scene_json["objects"])
    if not counts:
        return "No se detectaron objetos en la escena."

    top = counts.most_common(8)
    object_list = ", ".join(f"{n} {id_to_name.get(cid, 'objeto')}" for cid, n in top)
    user_prompt = f"Objetos detectados (categoría y cantidad): {object_list}"
    return _generate(_DESCRIPTION_SYSTEM_PROMPT, user_prompt, max_new_tokens=120)


def _is_valid_interpretation(interpreted: dict, name_to_id: dict[str, int]) -> bool:
    if interpreted.get("type") not in QUESTION_TYPES:
        return False
    names = interpreted.get("class_en", [])
    if not isinstance(names, list) or not all(isinstance(n, str) and n in name_to_id for n in names):
        return False
    if interpreted["type"] == "relation" and interpreted.get("predicate") not in PREDICATES:
        return False
    return True


def answer_question_llm(scene_json: dict, question_record: dict, categories: list[dict]) -> dict:
    """question_record: only question_id/type/question are read from it for
    bookkeeping (preserved in the output, same as deterministic mode) --
    type/class_en/class_es are never passed to the LLM, only question_record
    ["question"]. Returns a questionAnswerLLM dict: everything
    answer_question returns, plus "interpreted_query" with the LLM's raw
    intermediate JSON (even when invalid/unsupported, for auditability)."""
    name_to_id = {c["name"]: c["id"] for c in categories}
    question_text = question_record["question"]
    interpreted = interpret_query(question_text, categories)

    if interpreted.get("type") == "description":
        text = generate_description_llm(scene_json, categories)
        objects = scene_json["objects"]
        result = {
            "question_id": question_record["question_id"],
            "type": question_record["type"],
            "question": question_text,
            "answer": text,
            "normalized_answer": text,
            "evidence": [o["id"] for o in objects],
            "confidence": min((o["confidence"] for o in objects), default=1.0),
        }
    elif _is_valid_interpretation(interpreted, name_to_id):
        synthetic_record = {
            "question_id": question_record["question_id"],
            "type": interpreted["type"],
            "question": question_text,
            "class_en": interpreted.get("class_en", []),
        }
        if "predicate" in interpreted:
            synthetic_record["predicate"] = interpreted["predicate"]
        result = answer_question(scene_json, synthetic_record, categories)
        result["question_id"] = question_record["question_id"]
        result["question"] = question_text
    else:
        result = unsupported_answer(question_record)

    result["interpreted_query"] = interpreted
    return result

In [ ]:
dev_answers_llm_by_image: dict[str, list[dict]] = {}
t0 = time.time()
for i, q in enumerate(dev_questions):
    image_id = q["image_file"].rsplit(".", 1)[0]
    scene = dev_scenes[image_id]
    with tracer.step(image_id, "qa_llm", {"question_id": q["question_id"], "type": q["type"]}):
        answer = answer_question_llm(scene, q, categories)
    dev_answers_llm_by_image.setdefault(image_id, []).append(answer)
    if (i + 1) % 15 == 0 or i == len(dev_questions) - 1:
        print(f"  [{i+1}/{len(dev_questions)}] {time.time()-t0:.0f}s")

total_llm = sum(len(v) for v in dev_answers_llm_by_image.values())
unsupported_llm = sum(1 for v in dev_answers_llm_by_image.values() for a in v if a["normalized_answer"] == "unsupported")
print(f"Respuestas LLM generadas: {total_llm} ({unsupported_llm} unsupported)")

In [ ]:
# Integra questions/questions_llm en cada escena de desarrollo -- a partir
# de aca dev_scenes[image_id] ya tiene el SceneJSON completo y valido.
for image_id, scene in dev_scenes.items():
    scene["questions"] = dev_answers_by_image.get(image_id, [])
    scene["questions_llm"] = dev_answers_llm_by_image.get(image_id, [])

## Etapa 8 — Evaluación cuantitativa y proceso experimental

`detection_f1`/`mask_f1` comparten la misma política LVIS-aware (categoría positiva / `neg_category_ids` / `not_exhaustive_category_ids` / no verificada, tomada literalmente del `README.md` del dataset) a través de un helper común `_lvis_aware_f1`, parametrizado por la función de IoU (bbox o máscara). `scene_json_consistency` valida schema + RLE decodificable + referencias de ids. `qa_accuracy` calcula exact-match contra referencias de ground truth, separado por tipo.

In [ ]:
from typing import Callable

import jsonschema
import numpy as np
from PIL import Image, ImageDraw



def _lvis_aware_f1(
    predictions: list[dict],
    ground_truth: dict,
    iou_fn: Callable[[dict, dict], float],
    iou_threshold: float,
) -> dict:
    """Shared policy for detection_f1/mask_f1: a category_id is positive for
    an image if it has GT annotations there. Positive + not_exhaustive:
    unmatched predictions are ignored (not FP). Positive, not
    not_exhaustive: unmatched predictions are FP. In neg_category_ids:
    every prediction is FP (no GT to miss, so no FN). Neither positive nor
    negative (unverified): predictions are ignored entirely, since absence
    of annotations doesn't prove the object is absent. iou_fn(pred, gt_ann)
    returns the overlap metric (bbox or mask IoU) between the two."""
    gt_by_image: dict[int, list[dict]] = {}
    for ann in ground_truth["annotations"]:
        gt_by_image.setdefault(ann["image_id"], []).append(ann)

    preds_by_image: dict[int, list[dict]] = {}
    for pred in predictions:
        preds_by_image.setdefault(pred["image_id"], []).append(pred)

    tp = fp = fn = 0
    ignored_predictions = 0

    for image in ground_truth["images"]:
        image_id = image["id"]
        gts = gt_by_image.get(image_id, [])
        preds = preds_by_image.get(image_id, [])
        neg_ids = set(image.get("neg_category_ids", []))
        not_exh_ids = set(image.get("not_exhaustive_category_ids", []))
        positive_ids = {ann["category_id"] for ann in gts}

        gt_by_category: dict[int, list[dict]] = {}
        for ann in gts:
            gt_by_category.setdefault(ann["category_id"], []).append(ann)
        preds_by_category: dict[int, list[dict]] = {}
        for pred in preds:
            preds_by_category.setdefault(pred["category_id"], []).append(pred)

        all_category_ids = positive_ids | neg_ids | set(preds_by_category.keys())
        for category_id in all_category_ids:
            cat_gts = gt_by_category.get(category_id, [])
            cat_preds = sorted(preds_by_category.get(category_id, []), key=lambda p: -p["confidence"])

            if category_id in positive_ids:
                matched_gt = [False] * len(cat_gts)
                for pred in cat_preds:
                    best_iou, best_idx = 0.0, -1
                    for idx, ann in enumerate(cat_gts):
                        if matched_gt[idx]:
                            continue
                        iou = iou_fn(pred, ann)
                        if iou > best_iou:
                            best_iou, best_idx = iou, idx
                    if best_iou >= iou_threshold:
                        matched_gt[best_idx] = True
                        tp += 1
                    elif category_id in not_exh_ids:
                        ignored_predictions += 1
                    else:
                        fp += 1
                fn += matched_gt.count(False)
            elif category_id in neg_ids:
                fp += len(cat_preds)
            else:
                ignored_predictions += len(cat_preds)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "ignored_predictions": ignored_predictions,
    }


def detection_f1(predictions: list[dict], ground_truth: dict, iou_threshold: float = 0.5) -> dict:
    """predictions: flat list of {image_id, category_id, confidence, bbox}.
    ground_truth: annotations.json dict (images with neg_category_ids /
    not_exhaustive_category_ids, categories, annotations). See
    _lvis_aware_f1 for the matching policy."""
    return _lvis_aware_f1(
        predictions, ground_truth,
        iou_fn=lambda pred, ann: bbox_iou(pred["bbox"], ann["bbox"]),
        iou_threshold=iou_threshold,
    )


def _polygon_to_mask(segmentation: list[list[float]], height: int, width: int) -> np.ndarray:
    image = Image.new("L", (width, height), 0)
    draw = ImageDraw.Draw(image)
    for polygon in segmentation:
        coords = [(polygon[i], polygon[i + 1]) for i in range(0, len(polygon), 2)]
        draw.polygon(coords, outline=1, fill=1)
    return np.array(image, dtype=bool)


def mask_f1(predictions: list[dict], ground_truth: dict, iou_threshold: float = 0.5) -> dict:
    """predictions: flat list of {image_id, category_id, confidence, mask}
    (mask = our RLE format, as produced by segmentation.mask_to_rle).
    ground_truth: annotations.json dict, with polygon segmentations
    (COCO-style list of [x1,y1,x2,y2,...]). Same LVIS-aware policy as
    detection_f1, but matching on mask IoU instead of bbox IoU."""
    image_hw = {img["id"]: (img["height"], img["width"]) for img in ground_truth["images"]}
    gt_mask_cache: dict[int, np.ndarray] = {}

    def iou_fn(pred: dict, ann: dict) -> float:
        pred_mask = rle_to_mask(pred["mask"])
        ann_id = ann["id"]
        if ann_id not in gt_mask_cache:
            h, w = image_hw[ann["image_id"]]
            gt_mask_cache[ann_id] = _polygon_to_mask(ann["segmentation"], h, w)
        gt_mask = gt_mask_cache[ann_id]

        intersection = np.logical_and(pred_mask, gt_mask).sum()
        union = np.logical_or(pred_mask, gt_mask).sum()
        return float(intersection) / float(union) if union else 0.0

    return _lvis_aware_f1(predictions, ground_truth, iou_fn=iou_fn, iou_threshold=iou_threshold)


def scene_json_consistency(scene_json: dict, schema: dict) -> dict:
    """Validates against scene_schema.json, RLE decodability, and id
    references (relations' subject/object, questions'/questions_llm's
    evidence must all point at known object ids). Returns {valid, errors}."""
    validator = jsonschema.Draft202012Validator(schema)
    errors = [e.message for e in validator.iter_errors(scene_json)]

    object_ids = {o["id"] for o in scene_json.get("objects", [])}
    for i, obj in enumerate(scene_json.get("objects", [])):
        label = obj.get("id", f"index {i}")
        try:
            counts_sum = sum(obj["mask"]["counts"])
            h, w = obj["mask"]["size"]
            if counts_sum != h * w:
                errors.append(
                    f"object {label}: RLE counts sum to {counts_sum}, expected height*width={h * w}"
                )
            mask = rle_to_mask(obj["mask"])
            if mask.shape != (h, w):
                errors.append(f"object {label}: decoded mask shape {mask.shape} != declared size {(h, w)}")
        except Exception as exc:
            errors.append(f"object {label}: RLE decode failed ({exc})")

    for i, rel in enumerate(scene_json.get("relations", [])):
        if rel.get("subject") not in object_ids:
            errors.append(f"relation {i}: subject {rel.get('subject')!r} is not a known object id")
        if rel.get("object") not in object_ids:
            errors.append(f"relation {i}: object {rel.get('object')!r} is not a known object id")

    for field in ("questions", "questions_llm"):
        for q in scene_json.get(field, []):
            for evidence_id in q.get("evidence", []):
                if evidence_id not in object_ids:
                    errors.append(
                        f"{field} {q.get('question_id')!r}: evidence {evidence_id!r} is not a known object id"
                    )

    return {"valid": len(errors) == 0, "errors": errors}


def qa_accuracy(predicted_answers: list[dict], reference_answers: list[dict]) -> dict:
    """predicted_answers: questionAnswer dicts (question_id, type,
    normalized_answer, ...), e.g. from qa_engine.answer_question.
    reference_answers: [{question_id, normalized_answer}, ...] built from
    ground truth (e.g. real counts from annotations.json) -- only
    questions with a known reference are scored. "unsupported" predictions
    are excluded from the accuracy denominator and reported separately,
    since they're an explicit abstention, not a wrong guess."""
    reference_by_id = {r["question_id"]: r["normalized_answer"] for r in reference_answers}

    correct = 0
    scored = 0
    unsupported = 0
    by_type_correct: dict[str, int] = {}
    by_type_scored: dict[str, int] = {}

    for pred in predicted_answers:
        question_id = pred["question_id"]
        if question_id not in reference_by_id:
            continue
        qtype = pred["type"]
        if pred["normalized_answer"] == "unsupported":
            unsupported += 1
            continue
        scored += 1
        by_type_scored[qtype] = by_type_scored.get(qtype, 0) + 1
        if pred["normalized_answer"] == reference_by_id[question_id]:
            correct += 1
            by_type_correct[qtype] = by_type_correct.get(qtype, 0) + 1

    return {
        "accuracy": correct / scored if scored else 0.0,
        "correct": correct,
        "scored": scored,
        "unsupported": unsupported,
        "by_type": {t: by_type_correct.get(t, 0) / n for t, n in by_type_scored.items()},
    }

In [ ]:
detection_result = detection_f1(dev_predictions, annotations, iou_threshold=0.5)
print(f"Deteccion -> F1={detection_result['f1']:.3f} P={detection_result['precision']:.3f} R={detection_result['recall']:.3f}")

all_dev_objects = [
    {**obj, "image_id": img_info["id"]}
    for img_info in annotations["images"]
    for obj in dev_scenes[Path(img_info["file_name"]).stem]["objects"]
]
mask_result = mask_f1(all_dev_objects, annotations, iou_threshold=0.5)
print(f"Mascaras  -> F1={mask_result['f1']:.3f} P={mask_result['precision']:.3f} R={mask_result['recall']:.3f}")

consistency_results = [scene_json_consistency(s, schema) for s in dev_scenes.values()]
num_consistent = sum(1 for r in consistency_results if r["valid"])
print(f"SceneJSON consistentes (desarrollo): {num_consistent}/{len(dev_scenes)}")

name_to_id = {c["name"]: c["id"] for c in categories}
from collections import Counter
gt_counts = Counter((a["image_id"], a["category_id"]) for a in annotations["annotations"])
reference = []
for q in dev_questions:
    if q["type"] not in ("count", "existence") or not q.get("class_en"):
        continue
    cat_id = name_to_id.get(q["class_en"][0])
    gt_n = gt_counts.get((q["image_id"], cat_id), 0) if cat_id is not None else 0
    reference.append({
        "question_id": q["question_id"],
        "normalized_answer": gt_n if q["type"] == "count" else ("yes" if gt_n > 0 else "no"),
    })

all_dev_answers = [a for answers in dev_answers_by_image.values() for a in answers]
qa_result = qa_accuracy(all_dev_answers, reference)
print(f"QA determinista -> accuracy={qa_result['accuracy']:.2f} by_type={qa_result['by_type']}")

### Proceso experimental — línea base (0.1) vs. alternativa (0.3)

Comparación controlada: mismo pipeline, mismo detector, solo cambia `confidence_threshold`. **Resultado ya documentado en `REPORTE.md` 4.7:** subir el umbral mejora precision y reduce la ambigüedad en QA, pero **empeora levemente el F1 de detección** (que pesa 75% del `PipelineScore` de la competencia, contra 10% de QA) — por eso el baseline 0.1 se mantiene como configuración final (celda de Etapa 9). Esta sección solo reproduce la comparación; no cambia la configuración usada para la entrega.

In [ ]:
if RUN_CONTROLLED_EXPERIMENT:
    _baseline_threshold = CONFIDENCE_THRESHOLD
    CONFIDENCE_THRESHOLD = 0.3
    print(f"Corriendo pipeline completa con confidence_threshold={CONFIDENCE_THRESHOLD}...")

    exp_predictions = detect_all(dev_image_ids, DEVELOPMENT_DIR / "images", lambda iid: dev_file_names[iid])
    exp_preds_by_image: dict[str, list[dict]] = {}
    for pred in exp_predictions:
        exp_preds_by_image.setdefault(pred["image_id"], []).append(pred)

    exp_scenes: dict[str, dict] = {}
    for img_info in annotations["images"]:
        image_id = Path(img_info["file_name"]).stem
        exp_scenes[image_id] = segment_and_enrich(
            image_id, DEVELOPMENT_DIR / "images" / img_info["file_name"],
            exp_preds_by_image.get(img_info["id"], []),
        )

    exp_detection = detection_f1(exp_predictions, annotations, iou_threshold=0.5)

    exp_answers = []
    for q in dev_questions:
        image_id = q["image_file"].rsplit(".", 1)[0]
        exp_answers.append(answer_question(exp_scenes[image_id], q, categories))
    exp_unsupported = sum(
        1 for a in exp_answers if a["normalized_answer"] == "unsupported" and a["type"] in ("position", "size", "relation")
    )

    print(f"Baseline (0.1): F1={detection_result['f1']:.3f}")
    print(f"Alternativa (0.3): F1={exp_detection['f1']:.3f}")
    print(f"Unsupported (position+size+relation) baseline vs 0.3: "
          f"{sum(1 for a in all_dev_answers if a['normalized_answer']=='unsupported' and a['type'] in ('position','size','relation'))} "
          f"vs {exp_unsupported}")

    CONFIDENCE_THRESHOLD = _baseline_threshold
    print(f"confidence_threshold restaurado a {CONFIDENCE_THRESHOLD} para la entrega final.")
else:
    print("RUN_CONTROLLED_EXPERIMENT=False, se saltea esta seccion.")

## Etapa 9 — Entrega final (`submission.csv` / `solution.csv`)

Se corre la pipeline completa (detección → segmentación → `scene_builder` → `properties`) sobre las **60 imágenes de test** (sin `questions`/`questions_llm`, no tienen benchmark asociado) y se combinan con las 20 escenas de desarrollo ya completas en un único archivo de 80 filas.

Como el PDF del enunciado llama al archivo `submission.csv` y la página de Kaggle lo llama `solution.csv` (discrepancia nunca resuelta por la cátedra), se escriben **ambos nombres con contenido idéntico**.

In [ ]:
import csv

test_images_meta = {}
with open(TEST_DIR / "test_images.csv") as f:
    for r in csv.DictReader(f):
        test_images_meta[r["image_id"]] = r

sample_rows = list(csv.DictReader(open(SAMPLE_SUBMISSION_PATH)))
test_ids = sorted(r["image_id"] for r in sample_rows if r["image_id"].startswith("test_"))

print(f"Procesando {len(test_ids)} imagenes de test (pipeline completa, sin QA)...")
t0 = time.time()
test_scenes: dict[str, dict] = {}
for i, image_id in enumerate(test_ids):
    meta = test_images_meta[image_id]
    image_path = TEST_DIR / "test_images" / meta["file_name"]
    image = Image.open(image_path).convert("RGB")

    with tracer.step(image_id, "detection", {"confidence_threshold": CONFIDENCE_THRESHOLD}):
        raw = detect_objects(image, VOCABULARY, confidence_threshold=CONFIDENCE_THRESHOLD)
    kept = suppress_duplicates(raw, iou_threshold=NMS_IOU_THRESHOLD)

    with tracer.step(image_id, "segmentation", {"num_boxes": len(kept)}):
        scene = segment_and_enrich(image_id, image_path, kept)
    test_scenes[image_id] = scene

    if (i + 1) % 10 == 0 or i == len(test_ids) - 1:
        print(f"  [{i+1}/{len(test_ids)}] {time.time()-t0:.0f}s, {len(scene['objects'])} objetos")

print(f"Escenas de test construidas: {len(test_scenes)}")

In [ ]:
rows = []
for image_id, scene in dev_scenes.items():
    rows.append((image_id, scene))
for image_id, scene in test_scenes.items():
    rows.append((image_id, scene))
rows.sort(key=lambda r: r[0])

validator = jsonschema.Draft202012Validator(schema)
num_invalid = 0
num_inconsistent = 0
for image_id, scene in rows:
    if list(validator.iter_errors(scene)):
        num_invalid += 1
        print(f"INVALID (schema) {image_id}")
    if not scene_json_consistency(scene, schema)["valid"]:
        num_inconsistent += 1
        print(f"INCONSISTENT {image_id}")

sample_ids = sorted(r["image_id"] for r in sample_rows)
our_ids = sorted(r[0] for r in rows)
assert our_ids == sample_ids, "El conjunto de image_id no coincide con sample_submission.csv"
assert len(rows) == 80, f"se esperaban 80 filas, hay {len(rows)}"

for filename in ("submission.csv", "solution.csv"):
    with open(OUTPUT_DIR / filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["image_id", "SceneJSON"])
        for image_id, scene in rows:
            writer.writerow([image_id, json.dumps(scene, ensure_ascii=False)])

print()
print(f"Filas: {len(rows)} (80 esperadas)")
print(f"Validas contra schema: {len(rows)-num_invalid}/{len(rows)}")
print(f"Consistentes (schema+RLE+referencias): {len(rows)-num_inconsistent}/{len(rows)}")
print(f"Escrito en {OUTPUT_DIR}/submission.csv y {OUTPUT_DIR}/solution.csv")

## Cierre

Las 9 etapas del enunciado quedaron implementadas y corridas con datos reales en esta notebook. Resumen de resultados — ver `REPORTE.md` del repositorio del grupo para el detalle completo de decisiones, experimentos y análisis de errores (incluidas las limitaciones identificadas, como la alucinación del LLM en la descripción generada).

- F1 de detección (baseline 0.1) y F1 de máscaras: ver salida de la celda de Etapa 8.
- `SceneJSON` 100% válido y consistente en las 80 filas de entrega.
- `submission.csv`/`solution.csv` guardados en `/kaggle/working/`, listos para enviar.
- Traza completa de la ejecución en `/kaggle/working/trace/pipeline.jsonl`.